# Medical Document Extraction — Development Playground

This notebook is for **step-by-step development and debugging without using the FastAPI API**.

Use it to test:

1. file validation  
2. text-PDF vs scanned-PDF/image detection  
3. quality assessment  
4. rendering and preprocessing  
5. OCR  
6. medical relevance detection  
7. document classification  
8. common field extraction  
9. lab / Pap smear / radiology extraction  
10. full stateless pipeline output  
11. batch testing on a folder of files  

Run this notebook from the **repository root** or set `REPO_ROOT` manually in the setup cell.


## 0. How to use

Recommended location in the repo:

```text
notebooks/extraction_development_playground.ipynb
```

Before running:

```bash
pip install -r requirements.txt
```

For OCR on images/scanned PDFs, make sure Tesseract is installed locally and Persian/English language packs are available if you use `eng+fas`.

This notebook does **not** call `/extract`, `/extract/file`, or any API route. It imports services directly from the `app/` package.


In [ ]:
# 1. Setup repository path
from pathlib import Path
import sys, os, json, tempfile, shutil, mimetypes, uuid
from dataclasses import asdict, is_dataclass
from pprint import pprint

# If the notebook is inside notebooks/, repo root is usually the parent folder.
# If this fails, manually set REPO_ROOT = Path('/path/to/Extract-text-from-medical-documents')
CWD = Path.cwd().resolve()
if (CWD / 'app').exists():
    REPO_ROOT = CWD
elif (CWD.parent / 'app').exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD  # change manually if needed

sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print('REPO_ROOT =', REPO_ROOT)
print('Python path contains repo:', str(REPO_ROOT) in sys.path)


In [ ]:
# 2. Import project services
# If this cell fails, check that you are running from the repository root and dependencies are installed.

from app.services.file_validation_service import FileValidationService, MIME_ALIASES, SUPPORTED
from app.services.document_analysis_service import DocumentAnalysisService
from app.services.quality_service import QualityService
from app.services.preprocessing_service import PreprocessingService
from app.services.ocr_service import OCRService
from app.services.relevance_service import RelevanceService
from app.services.classification_service import ClassificationService
from app.services.common_field_extractor import CommonFieldExtractor
from app.services.lab_extractor import LabExtractor
from app.services.pap_smear_extractor import PapSmearExtractor
from app.services.radiology_extractor import RadiologyExtractor
from app.services.confidence_service import ConfidenceService
from app.services.extraction_pipeline import ExtractionPipeline, ExtractionInput

try:
    from app.services.url_file_loader import load_url_to_tempfile, UrlFileLoadError
    URL_LOADER_AVAILABLE = True
except Exception as e:
    URL_LOADER_AVAILABLE = False
    print('URL loader unavailable:', e)

print('Imports OK')


In [ ]:
# 3. Choose input
# You can test a local file path, a file inside samples/raw, or a backend pre-signed URL.

INPUT_MODE = 'file_path'  # 'file_path' or 'file_url'

# Option A: local file path
# Put test PDFs/images in samples/raw/ and set SAMPLE_FILE manually if needed.
sample_candidates = []
for folder in [REPO_ROOT / 'samples' / 'raw', REPO_ROOT, Path('/mnt/data')]:
    if folder.exists():
        sample_candidates += list(folder.glob('*.pdf')) + list(folder.glob('*.jpg')) + list(folder.glob('*.jpeg')) + list(folder.glob('*.png')) + list(folder.glob('*.webp'))

SAMPLE_FILE = str(sample_candidates[0]) if sample_candidates else ''

# Option B: backend pre-signed URL. This is loaded directly with url_file_loader, not through the API.
FILE_URL = ''  # e.g. 'https://storage.example.com/temp/report.pdf?signature=...'

DOCUMENT_ID = 'notebook-doc-001'
REQUEST_ID = f'notebook-{uuid.uuid4()}'
DEBUG = True

print('Found sample candidates:', len(sample_candidates))
print('Selected SAMPLE_FILE:', SAMPLE_FILE)


In [ ]:
# 4. Helper functions
from IPython.display import display, Markdown, Image as IPyImage


def to_builtin(obj):
    # Convert dataclass/Pydantic/list/dict objects into JSON-serializable builtins.
    if obj is None:
        return None
    if hasattr(obj, 'model_dump'):
        return obj.model_dump()
    if hasattr(obj, 'dict'):
        return obj.dict()
    if is_dataclass(obj):
        return asdict(obj)
    if isinstance(obj, list):
        return [to_builtin(x) for x in obj]
    if isinstance(obj, tuple):
        return [to_builtin(x) for x in obj]
    if isinstance(obj, dict):
        return {k: to_builtin(v) for k, v in obj.items()}
    return obj


def show_json(obj, max_chars=20000):
    text = json.dumps(to_builtin(obj), ensure_ascii=False, indent=2)
    if len(text) > max_chars:
        text = text[:max_chars] + '
... TRUNCATED ...'
    display(Markdown(f"```json
{text}
```"))


def guess_mime(path: str):
    ext = Path(path).suffix.lower()
    if ext in SUPPORTED:
        return SUPPORTED[ext]
    return mimetypes.guess_type(path)[0] or 'application/octet-stream'


def preview_image(path, width=500):
    try:
        display(IPyImage(filename=str(path), width=width))
    except Exception as e:
        print('Cannot display image:', e)

print('Helpers ready')


## 1. Load input file

For backend URL testing, this cell downloads the URL directly using the repository URL loader, then returns a temporary file path. This still does **not** use the API.


In [ ]:
# 5. Resolve input path
TEMP_INPUT_DIR = None

if INPUT_MODE == 'file_url':
    assert URL_LOADER_AVAILABLE, 'URL loader is not available in this repository version.'
    assert FILE_URL, 'Set FILE_URL first.'
    TEMP_INPUT_DIR = tempfile.TemporaryDirectory(prefix='notebook_url_input_')
    try:
        loaded = load_url_to_tempfile(FILE_URL, output_dir=TEMP_INPUT_DIR.name)
    except UrlFileLoadError as e:
        raise RuntimeError(f'URL load failed: {e.code} - {e.message}')
    FILE_PATH = loaded.file_path
    FILE_NAME = loaded.file_name
    MIME_TYPE = loaded.mime_type
else:
    assert SAMPLE_FILE, 'Set SAMPLE_FILE to a PDF/image path.'
    FILE_PATH = str(Path(SAMPLE_FILE).resolve())
    FILE_NAME = Path(FILE_PATH).name
    MIME_TYPE = guess_mime(FILE_PATH)

print('FILE_PATH =', FILE_PATH)
print('FILE_NAME =', FILE_NAME)
print('MIME_TYPE =', MIME_TYPE)
print('File exists:', Path(FILE_PATH).exists(), 'size:', Path(FILE_PATH).stat().st_size if Path(FILE_PATH).exists() else None)

if Path(FILE_PATH).suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
    preview_image(FILE_PATH, width=450)


## 2. File validation

This checks extension, MIME type, file size, PDF readability/password protection, and image readability.


In [ ]:
# 6. File validation
file_validator = FileValidationService()
normalized_mime = MIME_ALIASES.get(MIME_TYPE or '', MIME_TYPE)
validation = file_validator.validate(FILE_PATH, FILE_NAME, normalized_mime)
show_json(validation)

assert validation.is_valid, f'File validation failed: {validation.reason}'


## 3. Document analysis

This detects whether the input is:

- text-based PDF
- scanned PDF
- image

Text PDFs should skip image quality/preprocessing and go directly to embedded text extraction.


In [ ]:
# 7. Document analysis
analysis_service = DocumentAnalysisService()
analysis = analysis_service.analyze(FILE_PATH, FILE_NAME)
show_json(analysis)


## 4. Quality assessment, rendering, and preprocessing

This section follows the same logic as Step 2 of the project:

- text PDF: skip image quality and preprocessing
- scanned PDF: render pages to temporary images, assess quality page by page
- image: assess image quality
- if needed, preprocess into temporary images


In [ ]:
# 8. Quality + rendering/preprocessing
quality_service = QualityService()
preprocessing_service = PreprocessingService()

WORK_DIR = Path(tempfile.mkdtemp(prefix='notebook_extract_work_'))
print('WORK_DIR =', WORK_DIR)

source_paths = []
ocr_paths = []
quality_before = None
quality_after = None
render_result = None
preprocess_result = None

if analysis.should_skip_image_quality_check:
    print('Text PDF detected: skipping image quality and preprocessing.')
    ocr_paths = [FILE_PATH]
else:
    if analysis.file_type == 'pdf':
        render_result = preprocessing_service.render_pdf_pages(FILE_PATH, max_pages=5, output_dir=str(WORK_DIR / 'rendered'))
        show_json({'render_result': render_result})
        source_paths = render_result.output_paths
        quality_before = quality_service.assess_many(source_paths)
    else:
        source_paths = [FILE_PATH]
        quality_before = quality_service.assess(FILE_PATH)

    print('Quality before preprocessing:')
    show_json(quality_before)

    if quality_before and quality_before.status == 'good_quality':
        ocr_paths = source_paths
        print('Good quality: using original/rendered image paths for OCR.')
    elif quality_before and quality_before.status == 'poor_quality' and not quality_before.is_fixable:
        print('Poor and not fixable. OCR is not recommended.')
        ocr_paths = source_paths
    else:
        preprocess_result = preprocessing_service.preprocess(
            FILE_PATH,
            max_pages=5 if analysis.file_type == 'pdf' else 1,
            output_dir=str(WORK_DIR / 'processed')
        )
        print('Preprocessing result:')
        show_json(preprocess_result)
        ocr_paths = preprocess_result.output_paths if preprocess_result.success else source_paths
        if ocr_paths:
            quality_after = quality_service.assess_many(ocr_paths) if len(ocr_paths) > 1 else quality_service.assess(ocr_paths[0])
            print('Quality after preprocessing:')
            show_json(quality_after)

print('OCR paths:')
for p in ocr_paths[:5]:
    print(' -', p)
    if Path(p).suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
        preview_image(p, width=350)


## 5. OCR / text extraction

This runs OCR or embedded PDF text extraction directly through `OCRService`.


In [ ]:
# 9. OCR
ocr_service = OCRService()

if analysis.should_skip_image_quality_check:
    ocr = ocr_service.extract_any(FILE_PATH, analysis)
else:
    ocr = ocr_service.extract_images_text(ocr_paths)

show_json({
    'success': ocr.success,
    'confidence': ocr.confidence,
    'text_length': len(ocr.text or ''),
    'pages': [{'page_number': p.page_number, 'confidence': p.confidence, 'text_length': len(p.text or ''), 'source_path': p.source_path} for p in ocr.pages],
    'warnings': getattr(ocr, 'warnings', []),
    'error': getattr(ocr, 'error', None),
})

print('
--- OCR TEXT PREVIEW ---')
print((ocr.text or '')[:4000])


## 6. Medical relevance and document classification

This checks whether the OCR/PDF text looks medical and classifies it as lab, Pap smear, radiology, or unknown medical.


In [ ]:
# 10. Relevance + classification
relevance_service = RelevanceService()
classification_service = ClassificationService()

relevance = relevance_service.check_from_text(ocr.text or '', FILE_NAME)
classification = classification_service.classify(ocr.text or '')

show_json({'relevance': relevance, 'classification': classification})


## 7. Common field extraction

Extracts patient/document metadata such as patient name, national ID hash, report date, center name, age, sex, and tracking number.


In [ ]:
# 11. Common fields
common_extractor = CommonFieldExtractor()
common_fields = common_extractor.extract_structured(ocr.text or '')
show_json(common_fields)


## 8. Lab extraction

This cell extracts lab result rows. It is useful for testing both:

- normal one-line rows
- Tavo-style column-major blocks
- false-positive guideline suppression


In [ ]:
# 12. Lab extraction
lab_extractor = LabExtractor()
lab_rows = lab_extractor.extract(ocr.text or '')
print('Lab result count:', len(lab_rows))
show_json(lab_rows[:50])

# Optional DataFrame view
try:
    import pandas as pd
    if lab_rows:
        display(pd.DataFrame(lab_rows).head(100))
except Exception as e:
    print('Pandas display skipped:', e)


## 9. Pap smear and radiology extraction

These are useful when the classifier detects Pap smear or radiology reports.


In [ ]:
# 13. Pap smear / radiology extraction
pap_extractor = PapSmearExtractor()
radiology_extractor = RadiologyExtractor()

pap_result = pap_extractor.extract(ocr.text or '')
radiology_result = radiology_extractor.extract(ocr.text or '')

show_json({'pap_smear': pap_result, 'radiology': radiology_result})


## 10. Full stateless pipeline output

This runs the same stateless extraction pipeline that the API uses internally, but **without calling any API endpoint**.


In [ ]:
# 14. Full ExtractionPipeline output
pipeline = ExtractionPipeline()

inp = ExtractionInput(
    file_path=FILE_PATH,
    file_name=FILE_NAME,
    mime_type=MIME_TYPE,
    document_id=DOCUMENT_ID,
    request_id=REQUEST_ID,
    debug=DEBUG,
)

response = pipeline.process(inp, debug=DEBUG)
show_json(response)


## 11. Save notebook output for one file

This writes a JSON result and OCR text file into `notebook_outputs/`.


In [ ]:
# 15. Save current output
OUT_DIR = REPO_ROOT / 'notebook_outputs'
(OUT_DIR / 'json').mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'ocr').mkdir(parents=True, exist_ok=True)

stem = Path(FILE_NAME).stem
json_path = OUT_DIR / 'json' / f'{stem}.json'
ocr_path = OUT_DIR / 'ocr' / f'{stem}.txt'

json_path.write_text(json.dumps(to_builtin(response), ensure_ascii=False, indent=2), encoding='utf-8')
ocr_path.write_text(ocr.text or '', encoding='utf-8')

print('Saved:', json_path)
print('Saved:', ocr_path)


## 12. Batch test a folder without API

This is a lightweight notebook version of evaluation. It runs `ExtractionPipeline` on all supported files in a folder and creates a summary table.


In [ ]:
# 16. Batch evaluation helper
from datetime import datetime

SUPPORTED_EXTS = {'.pdf', '.jpg', '.jpeg', '.png', '.webp'}

def run_folder(input_dir='samples/raw', limit=None, debug=False):
    input_dir = Path(input_dir)
    files = [p for p in sorted(input_dir.iterdir()) if p.suffix.lower() in SUPPORTED_EXTS]
    if limit:
        files = files[:limit]
    rows = []
    pipe = ExtractionPipeline()
    for i, path in enumerate(files, start=1):
        print(f'[{i}/{len(files)}] {path.name}')
        resp = pipe.process(ExtractionInput(
            file_path=str(path),
            file_name=path.name,
            mime_type=guess_mime(str(path)),
            document_id=path.stem,
            request_id=f'nb-batch-{uuid.uuid4()}',
            debug=debug,
        ), debug=debug)
        d = to_builtin(resp)
        common = d.get('common_fields') or {}
        extracted = d.get('extracted_data') or {}
        lab_rows = extracted.get('lab_results') or []
        rows.append({
            'filename': path.name,
            'status': d.get('status'),
            'document_type': d.get('document_type'),
            'confidence': d.get('confidence'),
            'ocr_success': (d.get('ocr') or {}).get('success'),
            'ocr_text_length': (d.get('ocr') or {}).get('text_length'),
            'patient_name_found': bool((common.get('patient_name') or {}).get('value')),
            'date_found': bool((common.get('date_of_test_or_report') or {}).get('value')),
            'national_id_hash_found': bool((common.get('national_id') or {}).get('hash')),
            'center_name_found': bool((common.get('center_name') or {}).get('value')),
            'age_found': bool((common.get('age') or {}).get('value')),
            'sex_found': bool((common.get('sex') or {}).get('value')),
            'lab_result_count': len(lab_rows),
            'error_codes': ';'.join([e.get('code','') for e in d.get('errors', [])]),
            'warning_codes': ';'.join([w.get('code','') for w in d.get('warnings', [])]),
        })
    try:
        import pandas as pd
        df = pd.DataFrame(rows)
        display(df)
        out_csv = REPO_ROOT / 'notebook_outputs' / f'batch_summary_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(out_csv, index=False)
        print('Saved summary:', out_csv)
        return df
    except Exception:
        show_json(rows)
        return rows

# Example:
# df = run_folder('samples/raw', limit=10, debug=False)


## 13. Parser sandbox: paste OCR text directly

Use this when OCR is already available and you only want to debug relevance/classification/common/lab extraction.


In [ ]:
# 17. Direct text parser sandbox
PASTED_OCR_TEXT = """
# Paste OCR text here, then run this cell.
"""

if PASTED_OCR_TEXT.strip() and 'Paste OCR text here' not in PASTED_OCR_TEXT:
    rel2 = relevance_service.check_from_text(PASTED_OCR_TEXT, 'pasted_text.txt')
    cls2 = classification_service.classify(PASTED_OCR_TEXT)
    common2 = common_extractor.extract_structured(PASTED_OCR_TEXT)
    labs2 = lab_extractor.extract(PASTED_OCR_TEXT)
    show_json({'relevance': rel2, 'classification': cls2, 'common_fields': common2, 'lab_rows': labs2})
else:
    print('Paste OCR text into PASTED_OCR_TEXT first.')


## 14. Cleanup temporary files

Run this when you are done. It removes only the notebook temporary working folder.


In [ ]:
# 18. Cleanup
try:
    if 'WORK_DIR' in globals() and Path(WORK_DIR).exists():
        shutil.rmtree(WORK_DIR)
        print('Removed WORK_DIR:', WORK_DIR)
    if TEMP_INPUT_DIR is not None:
        TEMP_INPUT_DIR.cleanup()
        print('Cleaned URL temp input')
except Exception as e:
    print('Cleanup warning:', e)
